# 청킹 → 배치 파이프라인 테스트

실제 로더에서 반환된 파일들로부터 청킹 → 배치 파이프라인의 전체 흐름을 테스트

## 테스트 목표

1. **실제 파일 로딩**: 로더 모듈을 통한 실제 파일 로딩
2. **청킹 파이프라인**: 파일별 문서 청킹
3. **배치 생성**: 토큰 기반 배치 생성
4. **전체 통합**: 임베딩 전 단계까지의 완전한 파이프라인 검증


In [ ]:
# 모듈 import 및 설정
import sys
from pathlib import Path
from typing import Dict, List, Tuple, Union, cast

from langchain_core.documents import Document
from omegaconf import DictConfig, OmegaConf

# 경로 설정 (상대 경로 사용)
project_root: Path = Path("../..").resolve()
source_root: Path = project_root / "src"

# sys.path에 프로젝트 루트 추가
sys.path.insert(0, str(project_root))

# config 로드
config_path: Path = source_root / "conf" / "config.yaml"
cfg: DictConfig = cast(DictConfig, OmegaConf.load(config_path))

# 모듈 import
from src.core.chunker.batch_processor import process_document_batches
from src.core.chunker.chunker import chunk_documents
from src.core.chunker.token_utils import MAX_TOKENS
from src.core.loader_router.loader import get_loader

print("✅ 모듈 import 완료")
print(f"프로젝트 루트: {project_root}")
print(f"Source 루트: {source_root}")
print(f"MAX_TOKENS: {MAX_TOKENS}")


In [ ]:
# 테스트 파일 경로 설정
def collect_test_files() -> Tuple[List[Path], Dict[str, List[str]]]:
    """테스트 파일들을 수집하고 확장자별로 분류"""
    print("=== 테스트 파일 경로 설정 ===")

    test_files_dir: Path = project_root / "notebooks" / "test-files"
    print(f"테스트 디렉토리: {test_files_dir}")
    print(f"디렉토리 존재: {test_files_dir.exists()}")

    if test_files_dir.exists():
        all_files: List[Path] = []
        supported_extensions: List[str] = [
            ".txt",
            ".md",
            ".pdf",
            ".docx",
            ".doc",
            ".hwp",
            ".org",
            ".rtf",
            ".xlsx",
            ".xls",
            ".csv",
            ".tsv",
            ".pptx",
            ".ppt",
        ]

        for ext in supported_extensions:
            files: List[Path] = list(test_files_dir.rglob(f"*{ext}"))
            all_files.extend(files)

        print(f"✅ 수집된 파일 수: {len(all_files)}개")

        file_by_ext: Dict[str, List[str]] = {}
        for file_path in all_files:
            ext: str = file_path.suffix.lower()
            if ext not in file_by_ext:
                file_by_ext[ext] = []
            file_by_ext[ext].append(str(file_path))

        print("\n확장자별 파일 수:")
        for ext, files_list in file_by_ext.items():
            print(f"  {ext}: {len(files_list)}개")

            for file_path_str in files_list[:3]:
                print(f"    - {Path(file_path_str).name}")
            if len(files_list) > 3:
                print(f"    ... 외 {len(files_list) - 3}개")

        return all_files, file_by_ext
    else:
        print("❌ 테스트 디렉토리가 존재하지 않습니다!")
        return [], {}


all_files: List[Path]
file_by_ext: Dict[str, List[str]]
all_files, file_by_ext = collect_test_files()


In [ ]:
# 로더를 통한 문서 로딩
def load_documents_from_files(
    files: List[Path], start_idx: int = 0, end_idx: int = 50
) -> List[List[Document]]:
    """파일들을 로더를 통해 로딩하여 Document 리스트로 변환"""
    print("=== 로더를 통한 문서 로딩 ===")

    if not files:
        print("❌ 로딩할 파일이 없습니다!")
        return []

    test_files: List[Path] = files[start_idx:end_idx]
    print(f"테스트할 파일 수: {len(test_files)}개")

    print("\n테스트할 파일 목록:")
    for i, file_path in enumerate(test_files, 1):
        i_typed: int = i
        print(f"  {i_typed}. {file_path.name}")

    docs_by_file: List[List[Document]] = []
    loaded_count: int = 0

    for file_path in test_files:
        try:
            loader = get_loader(str(file_path), cfg)
            if loader:
                docs: List[Document] = loader.load()
                docs_by_file.append(docs)
                loaded_count += 1
                print(f"✅ {file_path.name}: {len(docs)}개 문서 로딩")
            else:
                print(f"❌ {file_path.name}: 로더 생성 실패")
                empty_docs_loader: List[Document] = []
                docs_by_file.append(empty_docs_loader)
        except Exception as e:
            print(f"❌ {file_path.name}: 로딩 실패 - {str(e)}")
            empty_docs_exception: List[Document] = []
            docs_by_file.append(empty_docs_exception)

    print(f"\n✅ 총 {loaded_count}개 파일 로딩 완료")
    print(f"총 문서 수: {sum(len(docs) for docs in docs_by_file)}개")

    return docs_by_file


docs_by_file: List[List[Document]] = load_documents_from_files(all_files)


In [ ]:
# 청킹 파이프라인
def test_individual_chunking(docs_by_file: List[List[Document]], test_files: List[Path]) -> int:
    """각 파일을 해당 확장자로 개별 청킹 테스트"""
    print("=== 파일별 개별 청킹 테스트 ===")

    if not docs_by_file or not test_files:
        print("❌ 청킹할 파일이 없습니다!")
        return 0

    print(f"📋 청킹 설정: 각 파일을 해당 확장자별 설정으로 개별 처리")

    total_chunks: int = 0

    # 처음 num_files개 파일만 테스트
    for i in range(min(NUM_TEST_FILES, len(test_files), len(docs_by_file))):
        file: Path = test_files[i]
        file_docs: List[Document] = docs_by_file[i]
        ext: str = file.suffix.lower()

        if not file_docs:
            print(f"  {i + 1}. {file.name} ({ext}): 문서 없음")
            continue

        # 개별 파일을 해당 확장자로 청킹
        individual_chunks: List[Document] = list(chunk_documents([file_docs], ext))
        chunk_count: int = len(individual_chunks)
        total_chunks += chunk_count

        # 청크 크기 통계
        chunk_sizes: List[int] = [len(chunk.page_content) for chunk in individual_chunks]
        avg_size: float = sum(chunk_sizes) / len(chunk_sizes) if chunk_sizes else 0.0

        print(f"  {i + 1}. {file.name} ({ext}): {chunk_count}개 청크, 평균 {avg_size:.0f}자")

        # 첫 번째 청크 미리보기
        if individual_chunks:
            print(f"     첫 청크: {individual_chunks[0].page_content}...")

    print(f"✅ 총 {total_chunks}개 청크 생성 완료")
    return total_chunks


# 청킹 테스트 파라미터 설정
CHUNKING_START_IDX: int = 0  # 시작 위치
NUM_TEST_FILES: int = 7  # 테스트할 파일 수

# 청킹 파이프라인 실행
test_files: List[Path] = all_files[CHUNKING_START_IDX : CHUNKING_START_IDX + NUM_TEST_FILES]
selected_docs: List[List[Document]] = docs_by_file[
    CHUNKING_START_IDX : CHUNKING_START_IDX + NUM_TEST_FILES
]
chunk_count: int = test_individual_chunking(selected_docs, test_files)


In [ ]:
# 배치 생성 파이프라인 (청킹 결과 연결)
print("=== 배치 생성 파이프라인 ===")

if docs_by_file and chunk_count > 0:
    print(
        f"📋 배치 생성 설정: 이전 청킹 결과({chunk_count}개 청크)를 MAX_TOKENS({MAX_TOKENS}) 기준으로 배치화"
    )
    # 이전 청킹에서 사용한 동일한 파라미터 재사용
    test_files: List[Path] = all_files[CHUNKING_START_IDX : CHUNKING_START_IDX + NUM_TEST_FILES]
    selected_docs: List[List[Document]] = docs_by_file[
        CHUNKING_START_IDX : CHUNKING_START_IDX + NUM_TEST_FILES
    ]

    all_chunks: List[Document] = []
    representative_ext: str = ".txt"  # 기본값

    # 각 파일을 해당 확장자로 청킹해서 모든 청크 수집
    for i in range(min(NUM_TEST_FILES, len(test_files), len(selected_docs))):
        file: Path = test_files[i]
        file_docs: List[Document] = selected_docs[i]
        ext: str = file.suffix.lower()

        if i == 0:  # 첫 번째 파일의 확장자를 대표로 사용
            representative_ext = ext

        if file_docs:
            individual_chunks: List[Document] = list(chunk_documents([file_docs], ext))
            all_chunks.extend(individual_chunks)

    chunks: List[Document] = all_chunks
    print(f"청킹 결과: {len(chunks)}개 청크")

    if chunks:
        # 배치 처리 실행
        batch_count: int = 0
        batch_info_type = Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]
        for batch_info in process_document_batches(chunks, 0, 0, representative_ext, MAX_TOKENS):
            batch_count += 1
            if batch_count <= 3:  # 처음 3개만 출력
                batch: List[Document] = cast(List[Document], batch_info["batch"])
                metadata: Dict[str, Union[int, float, List[int]]] = cast(
                    Dict[str, Union[int, float, List[int]]], batch_info["metadata"]
                )
                print(f"  배치 {batch_count}: {len(batch)}개 청크")

        print(f"✅ 총 {batch_count}개 배치 생성 완료")
    else:
        print("❌ 청킹된 청크가 없습니다!")
else:
    print("❌ 이전 청킹 결과가 없습니다!")


In [ ]:
# 전체 파이프라인 검증
print("=== 전체 파이프라인 검증 ===")

if docs_by_file and "chunk_count" in locals() and "batch_count" in locals():
    print("✅ 전체 파이프라인 성공!")

    # 청킹에서 사용한 파일 범위 정보
    test_files: List[Path] = all_files[CHUNKING_START_IDX : CHUNKING_START_IDX + NUM_TEST_FILES]
    used_files_count: int = len(test_files)
    total_files_count: int = len(all_files)

    print(f"\n📊 파일 처리 범위:")
    print(f"  - 전체 수집 파일: {total_files_count}개")
    print(
        f"  - 실제 처리 파일: {used_files_count}개 (인덱스 {CHUNKING_START_IDX}-{CHUNKING_START_IDX + NUM_TEST_FILES - 1})"
    )

    print(f"\n🔄 파이프라인 단계별 결과:")
    print(f"  1. 파일 로딩: {len(all_files)}개 파일 전체 수집")
    print(
        f"  2. 청킹 대상: {used_files_count}개 파일 선택 (인덱스 {CHUNKING_START_IDX}-{CHUNKING_START_IDX + NUM_TEST_FILES - 1})"
    )

    # 청킹 대상 파일들의 문서 수 계산
    # 실제 청킹한 범위의 문서 수 계산
    selected_docs: List[List[Document]] = docs_by_file[
        CHUNKING_START_IDX : CHUNKING_START_IDX + min(NUM_TEST_FILES, len(test_files))
    ]
    chunking_target_docs: int = sum(len(docs) for docs in selected_docs)

    print(f"  3. 청킹 대상 문서 수: {chunking_target_docs}개")
    print(f"  4. 청크 수: {chunk_count}개")
    print(f"  5. 배치 수: {batch_count}개")

    # 효율성 지표
    print(f"\n📈 처리 효율성:")
    print(
        f"  - 대상 파일당 평균 문서: {chunking_target_docs / used_files_count if used_files_count > 0 else 0:.1f}개"
    )
    print(
        f"  - 문서당 평균 청크: {chunk_count / chunking_target_docs if chunking_target_docs > 0 else 0:.1f}개"
    )
    print(f"  - 배치당 평균 청크: {chunk_count / batch_count if batch_count > 0 else 0:.1f}개")

    print(f"\n🎯 임베딩 준비 완료!")
    print("다음 단계: 배치별로 임베딩 생성")
else:
    print("❌ 파이프라인 실행 실패!")
    print("필요한 변수들이 정의되지 않았습니다.")
